# **// Transcriptor**
## *Versión 1.2*

\> Este es un formato de documento interactivo tipo Jupyter Notebook pensado para la creación, automatización, organización y estructuración de las actas de las reuniones del equipo.

\> Este consta de brindarle los archivos de audios de la reunión dada, y de forma casi automática transcribe los audios de cada integrante con alineación incluida, los combina en un archivo centralizado y resume con IA (por token de Google Studio) dicho archivo.

> Este cuaderno está pensado para usarse dentro de Google Colab. Queda bajo su responsabilidad el uso del mismo por otros medios que no sea el recomendado.

# **> Instalación de las dependencias necesarias**

In [1]:
# creamos las carpetas
!mkdir audios
!mkdir "Transcripciones invididuales"

# instalamos WhisperX y dependencias de audio
!pip cache purge
!pip install git+https://github.com/m-bain/whisperX.git ffmpeg
!pip install -q -U google-genai pydantic

# **> Configuración inicial**

In [3]:
from datetime import datetime, timedelta
import locale
import pytz

SPRINT = "sprint-1"    # IMPORTANTE
REUNION = "1ra"       # IMPORTANTE

# como los audios invididuales de cada integrante tienen de nombre el nombre de usuario (de discord) de cada uno,
# esta lista remplaza esos nombres de usuario por los nombres de los integrantes
NOMBRES = {"usuario1": "nombreDelUsuario1", "usuario2": "nombreDelUsuario2", "usuario3": "nombreDelUsuario3", "usuario4": "nombreDelUsuario4"}


# obtenemos el tiempo
ahora = datetime.now(pytz.timezone('America/Argentina/Buenos_Aires'))

# meses
meses_espanol = ["", "enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

fecha_hoy = ahora.strftime(f"%d de {meses_espanol[ahora.month]} del %Y")

# en caso de retroceder la fecha, descomentar linea siguiente y cambiar: timedelta(days=n)
#fecha_hoy = (ahora - timedelta(days=1)).strftime(f"%d de {meses_espanol[ahora.month]} del %Y")

print(fecha_hoy)

# **> Ejecución del modelo para transcribir**

In [4]:
# @title
# importaciones
from google.colab import files
from pathlib import Path
from sys import exit
import gc
import torch
import whisperx


# configuración
audio_dir = Path(r"audios") # <--- cambiá por el nombre de la carpeta de los audios

device = "cuda"
batch_size = 16 # reduce si da error de memoria (out of memory)
compute_type = "float16" # cambiá a "int8" si es por CPU o si la GPU es pequeña

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# soluciona el error de mrd
_original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_load(*args, **kwargs)
torch.load = patched_load


# revisamos si las variables están definidas
#print(NOMBRES)


# transcribir con WhisperX
if not audio_dir.is_dir():
    print("> La carpeta de audios no existe")
    exit()

files_list = [file for file in audio_dir.iterdir() if file.is_file()]

files_contained = len(files_list)
if files_contained == 0:
    print("> La carpeta no contiene ningún archivo")
    exit()

print("> Cargando modelo de transcripción...")
model = whisperx.load_model("large-v3", device, compute_type=compute_type, language="es")

for audio_file in files_list:
    username = audio_file.stem[2:]

    print(f"\n> Transcribiendo \"{username}\"")
    audio = whisperx.load_audio(audio_file)
    result = model.transcribe(audio, batch_size=batch_size, language="es")

    # limpieza de memoria intermedia
    gc.collect(); torch.cuda.empty_cache(); #del model

    # alinear la transcripción (para que el texto calce perfecto con el tiempo)
    print("> Alineando texto...")
    model_a, metadata = whisperx.load_align_model(language_code="es", device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

    # limpieza de memoria intermedia
    gc.collect(); torch.cuda.empty_cache(); #del model_a

    # abrimos un archivo para guardar el contenido
    with open(f"./Transcripciones invididuales/Transcripción de {NOMBRES[username]} - {REUNION} reunión, {SPRINT}.txt", "w", encoding="utf-8") as f:
        print(f"> Guardando contenido en \"{f.name}\"...")

        f.write(f"Transcripción de voz de {NOMBRES[username]} ({username}) de la {REUNION} reunión del sprint {SPRINT}, realizada el día {fecha_hoy}\n\n")

        # obtenemos todos los segmentos
        segments = result["segments"]

        for i, segment in enumerate(segments):
            texto = segment["text"].strip()

            # obtenemos y formateamos el tiempo
            start_time = segment.get("start")
            end_time = segment.get("end")

            if start_time is not None and end_time is not None:
                # convertimos segundos a HH:MM:SS
                start_h = int(start_time // 3600)
                start_m = int((start_time % 3600) // 60)
                start_s = int(start_time % 60)
                end_h = int(end_time // 3600)
                end_m = int((end_time % 3600) // 60)
                end_s = int(end_time % 60)
                time_str = f"[{start_h:02d}:{start_m:02d}:{start_s:02d} - {end_h:02d}:{end_m:02d}:{end_s:02d}] "
            else:
                time_str = ""

            # si es la última línea no le pone coma al final, a las demás sí
            if i < len(segments) - 1:
                line = f"{time_str}{NOMBRES[username]}: {texto},\n"
            else:
                line = f"{time_str}{NOMBRES[username]}: {texto}\n"

            print(line, end="")
            f.write(line)

        print("\n> Archivo guardado")
        f.close()

del model
del model_a
print("\n> Transcripciones completadas")


# descarga de las transcripciones
print("\n> Comprimiendo...")
archivo_zip = "transcripciones.zip"

!zip -9 {archivo_zip} "Transcripciones invididuales"/*


# descarga de las transcripciones
print("\n> Descargando...")
files.download(archivo_zip)


# fin
print("\n> Operaciones completadas!!!")


# **> Conversión de la transcripción final**

In [5]:
# @title
transcriptions_list = [file for file in Path("./Transcripciones invididuales/").iterdir() if file.is_file()]
transcriptions_total = len(transcriptions_list)

if transcriptions_total == 0:
    print("> La carpeta no contiene ningún archivo")
    exit()

print(f"\n> Se encontraron {transcriptions_total} archivos de transcripción para mezclar.")

total_lines = []
print("\n> Mezclando...")

for file in transcriptions_list:
    with open(file, "r", encoding="utf-8") as f:
        # saltamos las 2 primeras líneas
        for _ in range(2):
            next(f)

        lines = f.read().split("\n")

        total_lines.extend(lines)
        f.close()


print(f"\t- {len(total_lines)} mezcladas")
print(f"\n> Ordenando...")

total_lines.sort()

FILENAME = f"./Transcripciones invididuales/Transcripción de la {REUNION} reunión del sprint {SPRINT}.txt"

with open(FILENAME, "w", encoding="utf-8") as f:
    f.write(f"Transcripción de la {REUNION} reunión del sprint {SPRINT}, realizada el día {fecha_hoy}\n\n")

    for line in total_lines:
        if not len(line) == 0:
            f.write(line)
            f.write("\n")

    f.close()

print(f"\n> Nuevo archivo de transcripción guardado")


# descarga del archivo
print("\n> Descargando...")

files.download(FILENAME)

# **> Resumen de la IA**



In [7]:
# @title
import re
import shutil
from google import genai
from google.colab import userdata
from google.genai import types


# clave 👀
client = genai.Client(api_key=userdata.get('api_key'))


# creamos el objeto de la IA para preguntarle y le subimos el archivo
chat = client.chats.create(model="gemini-3.5-flash")


# copiamos el archivo a un nombre temporal sin tildes/ñ
nombre_temporal = "transcripcion_temp.txt"
shutil.copy(FILENAME, nombre_temporal)

archivo = client.files.upload(
    file=nombre_temporal,
    config=types.UploadFileConfig(display_name=FILENAME)
)


# le preguntamo
response = chat.send_message([
    archivo,
    "podrías hacer un resumen de lo que se habló en la reunión? tenés como fuente la misma transcripción. otra cosa, solo te permito usar la negrita unicamente para titulos, subtitulos y etc, no para cuerpos de textos como parrafos. no busques nada de internet, solo basate en esta transcripción."
])

!rm -f {nombre_temporal}

# recibimos la response
output = response.text
print(output)


# exportando resultado
with open("output.txt", "w", encoding="utf-8") as f:
    f.write(output)
    f.close()


# salida
print("\n> Operación finalizada")


# descargamos
print("\n> Descargando...")
files.download("output.txt")

print("\n> Qué tenga buen día")